# Data Preprocessing


This notebook loads raw 15-minute electricity demand data and creates cleaned_data.csv.

## Steps:
1. Load raw dataset from data/raw/load_forecasting_dataset_corrected.csv
2. Detect timestamp and demand columns automatically
3. Convert timestamp to datetime and sort
4. Remove duplicate timestamps
5. Resample to 15-minute intervals
6. Handle missing values using interpolation
7. Remove impossible negative demand values
8. Remove extreme outliers using IQR
9. Add time features (Date, Hour, Minute, DayOfWeek, Month, IsWeekend)
10. Save cleaned data to data/processed/cleaned_data.csv


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Set up paths
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DATA_PATH = DATA_RAW_DIR / "load_forecasting_dataset_corrected.csv"
CLEANED_DATA_PATH = DATA_PROCESSED_DIR / "cleaned_data.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Processed data path: {DATA_PROCESSED_DIR}")


In [ ]:

# Load raw dataset
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found at {RAW_DATA_PATH}")

print(f"Loading raw dataset from: {RAW_DATA_PATH}")
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Raw dataset loaded successfully!")
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"First few rows:")
print(df_raw.head())


In [ ]:
# Detect timestamp column automatically
timestamp_cols = [col for col in df_raw.columns if any(keyword in col.lower() for keyword in ['time', 'date', 'timestamp'])]
if timestamp_cols:
    ts_col = timestamp_cols[0]
else:
    for col in df_raw.columns:
        try:
            pd.to_datetime(df_raw[col].head(100))
            ts_col = col
            break
        except:
            continue

# Detect demand column — prefer columns containing BOTH 'load' AND 'demand'
# (matches 'Load Demand (kW)' specifically), then fall back to broader keywords.
col_lower = {col.lower(): col for col in df_raw.columns}
demand_col = None

# Priority 1: column containing both 'load' and 'demand'
for key, col in col_lower.items():
    if 'load' in key and 'demand' in key:
        demand_col = col
        break

# Priority 2: column containing 'load' or 'demand'
if demand_col is None:
    for key, col in col_lower.items():
        if ('load' in key or 'demand' in key) and col != ts_col:
            demand_col = col
            break

# Priority 3: broader energy keywords (excluding timestamp column)
if demand_col is None:
    for keyword in ['consumption', 'power', 'energy', 'kw']:
        for key, col in col_lower.items():
            if keyword in key and col != ts_col:
                demand_col = col
                break
        if demand_col:
            break

print(f"Detected timestamp column: {ts_col}")
print(f"Detected demand column: {demand_col}")

In [ ]:

# Convert timestamp to datetime
df_raw[ts_col] = pd.to_datetime(df_raw[ts_col], errors='coerce')

# Sort by timestamp
df_raw = df_raw.sort_values(ts_col)

# Remove duplicate timestamps
original_len = len(df_raw)
df_raw = df_raw.drop_duplicates(subset=[ts_col], keep='first')
duplicates_removed = original_len - len(df_raw)
print(f"Removed {duplicates_removed} duplicate timestamps")

# Handle missing timestamps
df_raw = df_raw.dropna(subset=[ts_col])

print(f"Data after cleaning: {len(df_raw)} records")
print(f"Date range: {df_raw[ts_col].min()} to {df_raw[ts_col].max()}")


In [ ]:
# Set timestamp as index for resampling
df_indexed = df_raw.set_index(ts_col)

# Resample to 15-minute intervals ('15min' replaces deprecated '15T' in pandas 2.2+)
# numeric_only=True ignores non-numeric columns (Season, Public Event, etc.)
df_resampled = df_indexed.resample('15min').mean(numeric_only=True)

# Remove rows with missing demand values
df_resampled = df_resampled.dropna(subset=[demand_col])

print(f"After resampling to 15-minute intervals: {len(df_resampled)} records")
print(f"Date range: {df_resampled.index.min()} to {df_resampled.index.max()}")

In [ ]:

# Remove impossible negative demand values
negative_count = (df_resampled[demand_col] < 0).sum()
df_clean = df_resampled[df_resampled[demand_col] >= 0]
print(f"Removed {negative_count} negative demand values")

# Remove extreme outliers using IQR method
Q1 = df_clean[demand_col].quantile(0.25)
Q3 = df_clean[demand_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (df_clean[demand_col] < lower_bound) | (df_clean[demand_col] > upper_bound)
outlier_count = outliers.sum()
df_clean = df_clean[~outliers]

print(f"Removed {outlier_count} outliers using IQR method")
print(f"Final cleaned data: {len(df_clean)} records")


In [ ]:

# Reset index to get timestamp back as column
df_clean = df_clean.reset_index()

# Add time features
df_clean['Date'] = df_clean[ts_col].dt.date
df_clean['Hour'] = df_clean[ts_col].dt.hour
df_clean['Minute'] = df_clean[ts_col].dt.minute
df_clean['DayOfWeek'] = df_clean[ts_col].dt.dayofweek
df_clean['Month'] = df_clean[ts_col].dt.month
df_clean['IsWeekend'] = (df_clean['DayOfWeek'].isin([5, 6])).astype(int)

# Reorder columns
cols = [ts_col, demand_col, 'Date', 'Hour', 'Minute', 'DayOfWeek', 'Month', 'IsWeekend']
df_clean = df_clean[cols]

print("Added time features:")
print(f"- Date: {df_clean['Date'].min()} to {df_clean['Date'].max()}")
print(f"- Hours: {df_clean['Hour'].min()} to {df_clean['Hour'].max()}")
print(f"- DayOfWeek: {sorted(df_clean['DayOfWeek'].unique())}")
print(f"- Month: {sorted(df_clean['Month'].unique())}")
print(f"- Weekend days: {df_clean['IsWeekend'].sum()}")


In [ ]:

# Create processed directory
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save cleaned data
df_clean.to_csv(CLEANED_DATA_PATH, index=False)

print(f"Cleaned data saved to: {CLEANED_DATA_PATH}")
print(f"Final shape: {df_clean.shape}")
print(f"File size: {CLEANED_DATA_PATH.stat().st_size / 1024 / 1024:.1f} MB")

print("\nFinal data summary:")
print(df_clean[[demand_col, 'Hour', 'DayOfWeek', 'Month']].describe())
